# 01 — Bottom-up feedback cycle only

Questo notebook esegue **solo l'estensione bottom-up**. Non riesegue actor extraction, HLG extraction o LLG extraction iniziale.

Flusso:

1. legge i JSON già prodotti da `output/top_down/`;
2. ricostruisce gli oggetti `HighLevelGoals` e `LowLevelGoals` salvati dalla baseline;
3. esegue un pre-pass conservativo che elimina soltanto LLG sicuramente superflui, senza modificare il file top-down;
4. costruisce evidenze numerate e avvia il ciclo KAOS-lite discovery-first;
5. usa **Gemini** e le funzioni top-down originali per materializzare HLG/LLG scoperti o riparati;
6. usa il modello configurato su **Groq** per coverage, metadati KAOS e non-regression gate;
7. salva HLG/LLG finali e `kaosAnalysis` in `output/top_down_bottom_up/`;
8. lascia immutato il file top-down originale.

In questo modo il confronto sperimentale è davvero A/B: **top-down** contro **top-down + bottom-up**, a partire dallo stesso identico stato iniziale.

In [ ]:
from pathlib import Path
import json
import re
import sys
import time

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from Gemini_API.APIKeysManager import APIKeysManager
from groundtruth import ALL_GROUNDTRUTHS
import src.llm_clients as llm_clients
GROUNDTRUTH_NAMES = {item["name"] for item in ALL_GROUNDTRUTHS}

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
TOP_DOWN_BOTTOM_UP_DIR = OUTPUT_ROOT / "top_down_bottom_up"
ITERATIONS_ROOT = OUTPUT_ROOT / "bottom_up_iterations"

for path in (
    TOP_DOWN_DIR,
    TOP_DOWN_BOTTOM_UP_DIR,
    ITERATIONS_ROOT,
):
    path.mkdir(parents=True, exist_ok=True)

# Optional: insert explicit Path(...) values here to run only selected baselines.
# Leave empty to process every baseline registered in ALL_GROUNDTRUTHS.
BASELINE_FILE_OVERRIDES = []

# Maximum number of feedback iterations. By default use the orchestrator constant.
GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE = 25

# One remote request cannot keep the notebook busy forever.
REQUEST_TIMEOUT_SECONDS = 120.0
API_MAX_RETRIES = 1

print("Project root:", PROJECT_ROOT)
print("Top-down inputs:", TOP_DOWN_DIR)
print("Top-down + bottom-up outputs:", TOP_DOWN_BOTTOM_UP_DIR)
print("Gemini generator:", llm_clients.GEMINI_MODEL)
print("Gemini keys available:", APIKeysManager.count())
print("Groq evaluator:", llm_clients.GROQ_EVALUATOR_MODEL)

## 1. Individuazione degli output top-down

Il notebook usa tutti i file `*_top_down.json` presenti nella cartella della baseline, salvo override espliciti.

In [ ]:
if BASELINE_FILE_OVERRIDES:
    baseline_files = [Path(p) for p in BASELINE_FILE_OVERRIDES]
else:
    baseline_files = []
    for path in sorted(TOP_DOWN_DIR.glob("*_top_down.json")):
        with path.open("r", encoding="utf-8") as file:
            dataset_name = json.load(file).get("name")
        if dataset_name in GROUNDTRUTH_NAMES:
            baseline_files.append(path)

missing = [p for p in baseline_files if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing baseline files: {missing}")
if not baseline_files:
    raise FileNotFoundError(
        f"No *_top_down.json found in {TOP_DOWN_DIR}. "
        "Run 01_pipeline_execution_top_down_only.ipynb first."
    )

print(f"Found {len(baseline_files)} top-down baseline file(s):")
for p in baseline_files:
    print("-", p.name)

## 2. Caricamento dello stato iniziale

I file top-down contengono direttamente `structuredHighLevelGoals` e `structuredLowLevelGoals`. Il ciclo riusa quindi le relazioni padre-figlio realmente prodotte dalla pipeline top-down, senza mapping hard-coded. Anche le eventuali rigenerazioni HLG/LLG riusano le funzioni top-down e leggono modalità di prompting e stato del critic da `pipelineConfiguration`, senza dedurli dal nome del file.

In [ ]:
from src.data_model import HighLevelGoals, LowLevelGoals
from src.examples.shot_learning import ShotPromptingMode


def _infer_mode_from_name(path: Path) -> str:
    match = re.search(r"(?:^|_)(ZERO_SHOT|ONE_SHOT|FEW_SHOT)(?=_|$)", path.stem.upper())
    if not match:
        raise ValueError(f"Cannot infer prompting mode from {path.name}")
    return match.group(1)


def _stage_generation_config(
    config: dict,
    stage_name: str,
    baseline_file: Path,
) -> tuple[ShotPromptingMode, bool]:
    stage = config.get(stage_name) or {}
    mode_name = stage.get("promptingMode") or _infer_mode_from_name(
        baseline_file
    )
    try:
        mode = ShotPromptingMode[str(mode_name).upper()]
    except KeyError as exc:
        raise ValueError(
            f"Unsupported {stage_name} prompting mode {mode_name!r} in "
            f"{baseline_file.name}"
        ) from exc
    evaluator_ablation = not bool(stage.get("evaluator", True))
    return mode, evaluator_ablation


def load_top_down_state(baseline_file: Path):
    with baseline_file.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    description = payload["description"]
    config = payload.get("pipelineConfiguration") or {}
    hlg_mode, hlg_evaluator_ablation = _stage_generation_config(
        config, "highLevelGoals", baseline_file
    )
    llg_mode, llg_evaluator_ablation = _stage_generation_config(
        config, "lowLevelGoals", baseline_file
    )
    if hlg_mode != llg_mode:
        raise ValueError(
            "Bottom-up regeneration currently requires the same prompting "
            f"mode for HLG and LLG; found {hlg_mode.name} and "
            f"{llg_mode.name} in {baseline_file.name}."
        )
    if hlg_evaluator_ablation != llg_evaluator_ablation:
        raise ValueError(
            "Bottom-up regeneration currently requires the same critic "
            f"configuration for HLG and LLG in {baseline_file.name}."
        )

    structured_hlg = payload.get("structuredHighLevelGoals")
    structured_llg = payload.get("structuredLowLevelGoals")

    if structured_hlg is None or structured_llg is None:
        raise ValueError(
            f"{baseline_file.name} does not contain structured top-down goals. "
            "Run 01_pipeline_execution_top_down_only.ipynb first."
        )

    high_level_goals = HighLevelGoals.model_validate(structured_hlg)
    low_level_goals = LowLevelGoals.model_validate(structured_llg)

    return {
        "payload": payload,
        "description": description,
        "high_level_goals": high_level_goals,
        "low_level_goals": low_level_goals,
        "prompting_mode": hlg_mode,
        "llama_ablation": hlg_evaluator_ablation,
        "source_kind": "structured baseline fields",
        "mapped_input_file": None,
    }

## 3. Gemini generator, Groq evaluator e lifecycle dei client

Quando il Global Goal Evaluator richiede una correzione, l'orchestrator bottom-up richiama direttamente lo stesso generatore Gemini e lo stesso evaluator Groq usati dal top-down. Gemini crea e chiude un client per ogni richiesta; il client persistente Groq viene invece ricreato per ogni dataset e chiuso al termine.

In [ ]:
import importlib
from openai import OpenAI
from key import get_key_llama
import src.bottom_up.goal_cycle_orchestrator as _bottom_up_orchestrator
importlib.reload(_bottom_up_orchestrator)
from src.bottom_up.goal_cycle_orchestrator import (
    run_global_goal_cycle,
    DEFAULT_GLOBAL_CYCLE_MAX_ITERATIONS,
    MAX_HLG_REMOVALS_PER_ITERATION,
)
from src.self_critique.refine_response import MAX_ATTEMPTS, QUALITY_THRESHOLD


def _close_client_safely(client):
    if client is None:
        return
    try:
        client.close()
    except Exception as exc:
        print(f"Warning while closing client: {type(exc).__name__}: {exc}")


def reset_model_clients():
    # Gemini clients are short-lived inside ModelWrapper. Reset only its
    # key rotation, then recreate the persistent Groq evaluator client.
    APIKeysManager.reset()
    _close_client_safely(getattr(llm_clients, "llama", None))

    llm_clients.llama = OpenAI(
        api_key=get_key_llama(),
        base_url="https://api.groq.com/openai/v1",
        timeout=REQUEST_TIMEOUT_SECONDS,
        max_retries=API_MAX_RETRIES,
    )


def close_model_clients():
    _close_client_safely(getattr(llm_clients, "llama", None))
    llm_clients.llama = None


def bottom_up_model_configuration() -> dict:
    return {
        "generatorProvider": "Google Gemini",
        "generatorModel": llm_clients.GEMINI_MODEL,
        "evaluatorProvider": "Groq",
        "evaluatorModel": llm_clients.GROQ_EVALUATOR_MODEL,
        "maxIterations": GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE,
        "qualityThreshold": QUALITY_THRESHOLD,
        "maxReflectionAttempts": MAX_ATTEMPTS,
        "llgCleanupPrepass": True,
        "discoveryStrategy": "KAOS_LITE_EVIDENCE_FIRST",
        "additiveLlgPatch": True,
        "destructiveChangeNonRegressionGate": True,
        "newActorDiscovery": True,
        "maxHlgRemovalsPerIteration": MAX_HLG_REMOVALS_PER_ITERATION,
    }


def combined_output_file_for(baseline_file: Path) -> Path:
    stem = baseline_file.stem
    base = stem.removesuffix("_top_down")
    return TOP_DOWN_BOTTOM_UP_DIR / f"{base}_top_down_bottom_up.json"


def iteration_directory_for(baseline_file: Path) -> Path:
    return ITERATIONS_ROOT / baseline_file.stem.removesuffix("_top_down")

## 4. Esecuzione bottom-up

Ogni dataset viene processato in modo indipendente. Prima del ciclo completo viene eseguito un pre-pass branch-local: rimuove LLG solo con evidenza strutturata e confidenza alta, non applica eliminazioni se mancano capacità essenziali e non può svuotare una branch. L'output combinato parte da una copia del JSON top-down e aggiunge il pre-pass, il trace completo del ciclo e le collezioni finali. Il file top-down di partenza non viene mai sovrascritto.

In [ ]:
cycle_results = {}
cycle_errors = {}

max_iterations = (
    DEFAULT_GLOBAL_CYCLE_MAX_ITERATIONS
    if GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE is None
    else int(GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE)
)

for baseline_file in baseline_files:
    try:
        state = load_top_down_state(baseline_file)
    except Exception as exc:
        dataset_name = baseline_file.stem
        cycle_errors[dataset_name] = f"INPUT_LOAD_ERROR - {type(exc).__name__}: {exc}"
        print(f"\n[{dataset_name}] FAILED while loading top-down state: {exc}")
        continue

    baseline_payload = state["payload"]
    dataset_name = baseline_payload["name"]
    iteration_dir = iteration_directory_for(baseline_file)
    iteration_dir.mkdir(parents=True, exist_ok=True)
    # Remove only iteration artifacts for this exact dataset so a shorter
    # rerun cannot be contaminated by files from an older execution.
    for stale_iteration in iteration_dir.glob("iteration_*.json"):
        stale_iteration.unlink()
    stale_prepass = iteration_dir / "llg_cleanup_prepass.json"
    if stale_prepass.exists():
        stale_prepass.unlink()

    print(f"\n[{dataset_name}] starting bottom-up cycle")
    print("  input:", baseline_file.name)
    print("  state source:", state["source_kind"])
    print("  prompting mode:", state["prompting_mode"].name)
    print("  evaluator ablation:", state["llama_ablation"])
    print("  initial HLG:", len(state["high_level_goals"].goals))
    print("  initial LLG:", len(state["low_level_goals"].low_level_goals))

    started_at = time.perf_counter()
    try:
        reset_model_clients()
        result = run_global_goal_cycle(
            project_description=state["description"],
            initial_high_level_goals=state["high_level_goals"],
            initial_low_level_goals=state["low_level_goals"],
            mode=state["prompting_mode"],
            evaluator_ablation=state["llama_ablation"],
            evaluation_output_directory=iteration_dir,
            max_iterations=max_iterations,
        )
    except Exception as exc:
        cycle_errors[dataset_name] = f"{type(exc).__name__}: {exc}"
        # Persist an error artifact so every dataset still has a JSON
        # record even when an evaluator/generator fails mid-cycle.
        error_payload = dict(baseline_payload)
        error_payload.update({"baselineSourceFile": baseline_file.name,
                              "bottomUpInputSource": state["source_kind"],
                              "bottomUpModelConfiguration": bottom_up_model_configuration(),
                              "evaluationOutputDirectory": str(iteration_dir),
                              "bottomUpError": cycle_errors[dataset_name]})
        error_file = combined_output_file_for(baseline_file)
        with error_file.open("w", encoding="utf-8") as f:
            json.dump(error_payload, f, indent=4, ensure_ascii=False)
        print(f"  FAILED: {cycle_errors[dataset_name]} | saved: {error_file.name}")
        continue
    finally:
        close_model_clients()
        print(f"  client resources closed after {time.perf_counter() - started_at:.1f}s")

    cycle_results[dataset_name] = result

    combined_payload = dict(baseline_payload)
    combined_payload.update(
        {
            "baselineSourceFile": baseline_file.name,
            "bottomUpInputSource": state["source_kind"],
            "bottomUpModelConfiguration": bottom_up_model_configuration(),
            "bottomUpMappedInputFile": (
                str(state["mapped_input_file"]) if state["mapped_input_file"] else None
            ),
            "evaluationOutputDirectory": str(iteration_dir),
            "globalGoalCycle": result.model_dump(mode="json"),
            "finalHighLevelGoals": result.final_high_level_goals.model_dump(mode="json"),
            "finalLowLevelGoals": result.final_low_level_goals.model_dump(mode="json"),
            "kaosAnalysis": (
                result.kaos_analysis.model_dump(mode="json")
                if result.kaos_analysis is not None
                else None
            ),
        }
    )

    combined_file = combined_output_file_for(baseline_file)
    with combined_file.open("w", encoding="utf-8") as f:
        json.dump(combined_payload, f, indent=4, ensure_ascii=False)

    cleanup = result.llg_cleanup_prepass
    cleanup_removed = (
        cleanup.initial_low_level_goal_count - cleanup.final_low_level_goal_count
        if cleanup is not None
        else 0
    )
    print(
        f"  saved: {combined_file.name} | "
        f"pre-pass removed LLG={cleanup_removed}, "
        f"converged={result.converged}, "
        f"iterations={result.completed_iterations}, "
        f"total HLG={len(result.final_high_level_goals.goals)}, "
        f"total LLG={len(result.final_low_level_goals.low_level_goals)}"
    )
    for final_hlg in result.final_high_level_goals.goals:
        branch_llg_count = sum(
            llg.high_level_associated.name == final_hlg.name
            and llg.high_level_associated.actor.name == final_hlg.actor.name
            for llg in result.final_low_level_goals.low_level_goals
        )
        print(f"    branch {final_hlg.name!r}: LLG={branch_llg_count}")

## 5. Confronto strutturale rapido

Questa sezione mostra solo le cardinalità prima/dopo e lo stato di convergenza. **Precision, Recall e F1 non vengono calcolati qui**: sono volutamente delegati al terzo notebook, che legge i due output persistiti in modo indipendente.

In [ ]:
print("\nBottom-up execution summary")
print("---------------------------")
print(f"Inputs considered: {len(baseline_files)}")
print(f"Cycles completed: {len(cycle_results)}")
print(f"Cycles failed: {len(cycle_errors)}")
print(f"Output directory: {TOP_DOWN_BOTTOM_UP_DIR}")

for dataset_name, result in cycle_results.items():
    cleanup = result.llg_cleanup_prepass
    cleanup_removed = (
        cleanup.initial_low_level_goal_count - cleanup.final_low_level_goal_count
        if cleanup is not None
        else 0
    )
    print(
        f"- {dataset_name}: converged={result.converged}, "
        f"stop_reason={result.stop_reason}, "
        f"iterations={result.completed_iterations}, "
        f"final HLG={len(result.final_high_level_goals.goals)}, "
        f"final LLG={len(result.final_low_level_goals.low_level_goals)}, "
        f"pre-pass removed LLG={cleanup_removed}"
    )
    if result.warnings:
        print("  warnings:", "; ".join(result.warnings))

if cycle_errors:
    print("\nErrors:")
    for dataset_name, error in cycle_errors.items():
        print(f"- {dataset_name}: {error}")